# Berlin Tech & Data Job Market Analysis — Notebook 1: Data Collection, Cleaning & Merging
This notebook handles:
1. Fetching job postings from the **Bundesagentur für Arbeit (BA) Jobsuche API** (and full descriptions via the detail endpoint) as well as **Arbeitnow API**.
2. Extracting nested fields (city, salary ranges, dates) and normalizing German/English text.
3. Classifying roles, detecting technical skills via regex, and standardizing schemas across sources.
4. Exporting the merged, cleaned dataset (`Berlin_Tech_Data_Jobs_Merged_Analytics.xlsx` and `Graduate_Dashboard_Data.csv`).

In [1]:
import base64
from datetime import datetime
import json
import re
import time
import numpy as np
import pandas as pd
import requests

## 1. Fetch Raw Listings from Bundesagentur für Arbeit (BA) API

In [2]:
BA_SEARCH_URL = "https://rest.arbeitsagentur.de/jobboerse/jobsuche-service/pc/v6/jobs"
BA_DETAIL_URL = "https://rest.arbeitsagentur.de/jobboerse/jobsuche-service/pc/v4/jobdetails/{}"
BA_HEADERS = {
    "X-API-Key": "jobboerse-jobsuche",
    "User-Agent": "BerlinDataAnalyticsProject/1.0",
}

def fetch_ba_jobs(city="Berlin", keyword="Data", max_pages=5, page_size=50):
    all_jobs = []
    for page in range(1, max_pages + 1):
        params = {
            "wo": city,
            "was": keyword,
            "angebotsart": "1",
            "page": page,
            "size": page_size,
        }
        resp = requests.get(BA_SEARCH_URL, headers=BA_HEADERS, params=params, timeout=30)
        resp.raise_for_status()
        data = resp.json()

        if page == 1:
            print(f"Total matching BA results: {data.get('maxErgebnisse')}")

        jobs = data.get("ergebnisliste", [])
        if not jobs:
            break
        all_jobs.extend(jobs)
        time.sleep(0.5)

    return all_jobs

raw_ba_jobs = fetch_ba_jobs(city="Berlin", keyword="Data", max_pages=5, page_size=50)
timestamp = datetime.now().strftime("%Y_%m_%d")
raw_snapshot_path = f"raw_berlin_data_jobs_{timestamp}.json"

with open(raw_snapshot_path, "w", encoding="utf-8") as f:
    json.dump(raw_ba_jobs, f, ensure_ascii=False, indent=2)

print(f"Saved {len(raw_ba_jobs)} raw BA records to {raw_snapshot_path}")

Total matching BA results: 700
Saved 250 raw BA records to raw_berlin_data_jobs_2026_11_22.json


## 2. Fetch Full Job Descriptions (BA Detail Endpoint) & Arbeitnow API

In [3]:
FETCH_FULL_DESCRIPTIONS = True  # Set False to skip detail endpoint calls

def fetch_ba_description(refnr):
    if not isinstance(refnr, str) or not refnr:
        return ""
    encoded_ref = base64.b64encode(refnr.encode("utf-8")).decode("utf-8")
    try:
        r = requests.get(BA_DETAIL_URL.format(encoded_ref), headers=BA_HEADERS, timeout=20)
        if r.status_code == 200:
            return r.json().get("stellenangebotsBeschreibung", "") or ""
    except requests.RequestException:
        pass
    return ""

def fetch_arbeitnow_berlin_jobs(max_pages=5):
    url = "https://www.arbeitnow.com/api/job-board-api"
    records = []
    for page in range(1, max_pages + 1):
        try:
            r = requests.get(url, params={"page": page}, timeout=20)
            r.raise_for_status()
            items = r.json().get("data", [])
            if not items:
                break
            for item in items:
                loc = str(item.get("location", ""))
                if "berlin" in loc.lower():
                    records.append(item)
            time.sleep(0.4)
        except requests.RequestException:
            break
    return records

## 3. Cleaning, Classification, Skill Extraction & Standardization Functions

In [4]:
SKILL_PATTERNS = {
    "Python": r"\bpython\b",
    "SQL": r"\bsql\b",
    "R": r"\br\b",
    "Power BI": r"\bpower\s*bi\b",
    "Tableau": r"\btableau\b",
    "Excel": r"\bexcel\b",
    "AWS": r"\baws\b",
    "Azure": r"\bazure\b",
    "GCP": r"\b(gcp|google cloud)\b",
    "Spark": r"\bspark\b",
    "dbt": r"\bdbt\b",
    "Airflow": r"\bairflow\b",
    "Snowflake": r"\bsnowflake\b",
    "SAP": r"\bsap\b",
    "Machine Learning / AI": r"\b(machine learning|ml|ai|ki)\b",
    "ETL / Data Warehouse": r"\b(etl|dwh|data warehouse)\b",
}

def extract_location(loc):
    if isinstance(loc, list) and loc:
        loc = loc[0]
    if isinstance(loc, dict):
        adresse = loc.get("adresse", {})
        if isinstance(adresse, dict):
            return adresse.get("ort") or adresse.get("region") or "Berlin"
    return "Berlin"

def classify_category(title):
    if not isinstance(title, str):
        return "Other Tech & Data"
    t = re.sub(r"[-/]+", " ", title.lower())
    t = re.sub(r"\s+", " ", t).strip()
    if any(k in t for k in ["data engineer", "datenengineer"]) or re.search(r"\betl\b", t):
        return "Data Engineer"
    if any(k in t for k in ["data scientist", "datascientist", "machine learning"]) or re.search(r"\b(ml|ki|ai)\b", t):
        return "Data Scientist & AI"
    if any(k in t for k in ["data analyst", "datenanalyst", "analytics", "datenanalyse", "business intelligence"]) or re.search(r"\bbi\b", t):
        return "Data Analyst & BI"
    if any(k in t for k in ["cloud", "devops", "sre", "kubernetes"]):
        return "Cloud & DevOps"
    if "backend" in t or "back end" in t:
        return "Backend Engineer"
    if "frontend" in t or "front end" in t:
        return "Frontend Engineer"
    if "full stack" in t or "fullstack" in t:
        return "Full Stack Engineer"
    if "product manager" in t or "product owner" in t:
        return "Product Management"
    if any(k in t for k in ["head of", "director", "vp ", "engineering manager", "cto"]):
        return "Engineering Leadership"
    if any(k in t for k in ["software", "developer", "entwickler", "it ", "system"]):
        return "Software & IT"
    return "Other Tech & Data"

def extract_skills(text):
    t = str(text).lower()
    matched = [skill for skill, pat in SKILL_PATTERNS.items() if re.search(pat, t)]
    return ", ".join(matched) if matched else "Not Specified"

def detect_language(text):
    t = str(text).lower()
    has_de = any(w in t for w in ["und", "für", "wir", "kenntnisse", "erfahrung", "m/w/d", "w/m/d"])
    has_en = any(w in t for w in ["and", "with", "experience", "skills", "team", "f/m/d"])
    if has_de and has_en:
        return "German & English"
    if has_de:
        return "German"
    if has_en:
        return "English"
    return "Not Specified"

def clean_employment(val):
    s = str(val).lower()
    if any(k in s for k in ["werkstudent", "working student"]):
        return "Working Student"
    if any(k in s for k in ["praktikum", "intern"]):
        return "Internship"
    if "teilzeit" in s or "part" in s:
        return "Part-Time"
    if "befristet" in s and "unbefristet" not in s:
        return "Fixed-Term Contract"
    if any(k in s for k in ["full", "unbefristet", "true"]):
        return "Full-Time"
    return "Not Specified"

def clean_experience(title_or_exp):
    s = str(title_or_exp).lower()
    if any(k in s for k in ["lead", "staff", "exec", "management", "head", "director", "principal"]):
        return "Lead / Management"
    if "senior" in s:
        return "Senior"
    if "mid" in s:
        return "Mid-Level"
    if any(k in s for k in ["junior", "intern", "student", "entry", "werkstudent", "praktikum", "berufseinstieg"]):
        return "Entry-Level / Student"
    return "Not Specified"

def format_ba_salary(row):
    von, bis, fix = row.get("gehaltsspanneVon"), row.get("gehaltsspanneBis"), row.get("festgehalt")
    if pd.notna(von) and pd.notna(bis):
        return f"€{int(von):,} - €{int(bis):,}".replace(",", ".")
    if pd.notna(fix):
        return f"€{int(fix):,}".replace(",", ".")
    return "Not Specified"

def parse_annual_eur(s):
    if not isinstance(s, str) or s == "Not Specified":
        return pd.Series([np.nan, np.nan, np.nan])
    nums = [float(x.replace(".", "")) for x in re.findall(r"€([\d\.]+)", s)]
    if not nums or max(nums) < 20000:
        return pd.Series([np.nan, np.nan, np.nan])
    s_min = nums[0]
    s_max = nums[1] if len(nums) >= 2 else nums[0]
    return pd.Series([s_min, s_max, (s_min + s_max) / 2])

## 4. Transform, Merge Sources & Export Clean Datasets

In [5]:
# Transform Bundesagentur für Arbeit records
df_ba_raw = pd.DataFrame(raw_ba_jobs).drop_duplicates(subset=["referenznummer"]).copy()

if FETCH_FULL_DESCRIPTIONS:
    print(f"Fetching full descriptions for {len(df_ba_raw)} BA jobs...")
    descs = []
    for idx, ref in enumerate(df_ba_raw["referenznummer"], start=1):
        descs.append(fetch_ba_description(ref))
        if idx % 25 == 0:
            print(f"  Processed {idx}/{len(df_ba_raw)}...")
        time.sleep(0.35)
    df_ba_raw["Description"] = descs
else:
    df_ba_raw["Description"] = ""

df_ba_clean = pd.DataFrame({
    "Category": df_ba_raw["stellenangebotsTitel"].apply(classify_category),
    "Job": df_ba_raw["stellenangebotsTitel"],
    "Company": df_ba_raw.get("firma", "Not Specified"),
    "Location": df_ba_raw["stellenlokationen"].apply(extract_location),
    "Description": df_ba_raw["Description"],
    "Skills": (df_ba_raw["stellenangebotsTitel"].fillna("") + " " + df_ba_raw["Description"].fillna("")).apply(extract_skills),
    "Experience": df_ba_raw["stellenangebotsTitel"].apply(clean_experience),
    "Language": (df_ba_raw["stellenangebotsTitel"].fillna("") + " " + df_ba_raw["Description"].fillna("")).apply(detect_language),
    "Salary": df_ba_raw.apply(format_ba_salary, axis=1),
    "Employment": df_ba_raw.get("arbeitszeitVollzeit", "").apply(lambda x: "Full-Time" if x is True else "Not Specified"),
    "Date": pd.to_datetime(df_ba_raw["datumErsteVeroeffentlichung"], errors="coerce").dt.strftime("%d.%m.%Y"),
    "Source": "Bundesagentur für Arbeit API"
})

# Transform Arbeitnow API records
raw_an_jobs = fetch_arbeitnow_berlin_jobs(max_pages=5)
if raw_an_jobs:
    df_an_raw = pd.DataFrame(raw_an_jobs).drop_duplicates(subset=["slug"]).copy()
    clean_desc = df_an_raw["description"].fillna("").apply(lambda x: re.sub(r"<[^>]+>", " ", x))
    df_an_clean = pd.DataFrame({
        "Category": df_an_raw["title"].apply(classify_category),
        "Job": df_an_raw["title"],
        "Company": df_an_raw["company_name"],
        "Location": df_an_raw["location"],
        "Description": clean_desc,
        "Skills": (df_an_raw["title"].fillna("") + " " + clean_desc).apply(extract_skills),
        "Experience": df_an_raw["title"].apply(clean_experience),
        "Language": (df_an_raw["title"].fillna("") + " " + clean_desc).apply(detect_language),
        "Salary": "Not Specified",
        "Employment": df_an_raw.get("job_types", "").apply(clean_employment),
        "Date": pd.to_datetime(df_an_raw["created_at"], unit="s", errors="coerce").dt.strftime("%d.%m.%Y"),
        "Source": "Arbeitnow API"
    })
    df_merged = pd.concat([df_ba_clean, df_an_clean], ignore_index=True)
else:
    df_merged = df_ba_clean.copy()

# Deduplicate across sources by Job + Company
df_merged.drop_duplicates(subset=["Job", "Company"], inplace=True)

# Enrich with standardized analytics columns
df_merged["Employment_Clean"] = df_merged["Employment"].apply(clean_employment)
df_merged["Experience_Clean"] = df_merged["Experience"].apply(clean_experience)
df_merged[["Salary_Min_EUR", "Salary_Max_EUR", "Salary_Mid_EUR"]] = df_merged["Salary"].apply(parse_annual_eur)
df_merged["Is_Core_Data"] = df_merged["Category"].isin(["Data Scientist & AI", "Data Analyst & BI", "Data Engineer"])

# Build Summary & Skill Matrix sheets
analytics_summary = (
    df_merged.groupby("Category", as_index=False)
    .size()
    .rename(columns={"size": "Job Count"})
    .sort_values("Job Count", ascending=False)
)
analytics_summary["Share (%)"] = (analytics_summary["Job Count"] / len(df_merged) * 100).round(1)

combined_corpus = (df_merged["Job"].fillna("") + " " + df_merged["Description"].fillna("")).str.lower()
skill_rows = []
for skill, pat in SKILL_PATTERNS.items():
    cnt = int(combined_corpus.apply(lambda txt: bool(re.search(pat, txt))).sum())
    skill_rows.append({"Skill": skill, "Job Count": cnt, "Share (%)": round(cnt / len(df_merged) * 100, 1)})
skill_matrix = pd.DataFrame(skill_rows).sort_values("Job Count", ascending=False)

# Export to Excel & CSV
excel_out = "Berlin_Tech_Data_Jobs_Merged_Analytics.xlsx"
with pd.ExcelWriter(excel_out, engine="openpyxl") as writer:
    df_merged.to_excel(writer, sheet_name="Merged_Jobs", index=False)
    analytics_summary.to_excel(writer, sheet_name="Analytics_Summary", index=False)
    skill_matrix.to_excel(writer, sheet_name="Skill_Matrix", index=False)

csv_out = "Graduate_Dashboard_Data.csv"
df_merged.to_csv(csv_out, index=False, encoding="utf-8-sig")

print(f"Successfully exported {len(df_merged)} merged records to {excel_out} and {csv_out}")

Fetching full descriptions for 249 BA jobs...
  Processed 25/249...
  Processed 50/249...
  Processed 75/249...
  Processed 100/249...
  Processed 125/249...
  Processed 150/249...
  Processed 175/249...
  Processed 200/249...
  Processed 225/249...
Successfully exported 330 merged records to Berlin_Tech_Data_Jobs_Merged_Analytics.xlsx and Graduate_Dashboard_Data.csv
